# AI Greeting Testing Notebook

Testing notebook for the `ai_greeting` chain which generates personalized course greetings in Indonesian.

**Chain Function:** `generate_greeting(student_name, course_metadata)`

## 1. Setup

Import dependencies and load environment variables.

In [1]:
import os
import sys
from dotenv import load_dotenv

# Add parent directory to path for imports
sys.path.insert(0, os.path.abspath('../..'))

load_dotenv()
print("Environment loaded")

Environment loaded


In [2]:
from ai_chains.chains import ai_greeting
from utils.content_loader import load_course_metadata

print("Modules imported")

Modules imported


## 2. Chain Configuration

Display current prompt template and LLM settings.

In [3]:
# Display prompt template
with open('../../ai_chains/prompts/ai_greeting.yaml', 'r', encoding='utf-8') as f:
    print("Prompt Template:")
    print(f.read())

Prompt Template:
template: |
  Kamu adalah AI tutor Maguru yang semangat dan ramah. Buat sapaan pembuka yang PERSONAL dan MENARIK untuk siswa Indonesia.

  Nama siswa: {student_name}
  Judul kursus: {course_title}
  Yang akan dipelajari: {learning_objectives}

  Panduan sapaan:
  - SEBUT nama siswa minimal 1x di kalimat (bukan di awal saja)
  - JELASKAN mengapa kursus ini menarik untuk dipelajari
  - TANYAKAN pertanyaan terbuka untuk memulai percakapan
  - GUNAKAN bahasa yang santai tapi semangat (bukan kaku)

  PANJANG: 2-4 kalimat saja. Jangan bertele-tele.
input_variables: [student_name, course_title, learning_objectives]



In [4]:
# Show LLM configuration
print("LLM Configuration:")
print(f"  OpenRouter Model: {os.getenv('OPENROUTER_MODEL', 'google/gemma-7b-it')}")
print(f"  Z.AI Model: {os.getenv('ZAI_MODEL', 'glm-4.7')}")
print(f"  OpenRouter Key: {'Set' if os.getenv('OPENROUTER_API_KEY') else 'Not set'}")
print(f"  Z.AI Key: {'Set' if os.getenv('ZAI_API_KEY') else 'Not set'}")

LLM Configuration:
  OpenRouter Model: arcee-ai/trinity-mini:free
  Z.AI Model: glm-4.7
  OpenRouter Key: Set
  Z.AI Key: Set


## 3. Load Test Data

Load course metadata from the python_basics course.

In [5]:
# Load course metadata
course_data = load_course_metadata("python_basics")

# Check if data loaded successfully
if course_data is None:
    print("Warning: Course metadata not found, using fallback data")
    course_data = {
        "id": "python_basics",
        "title": "Python Basics for Beginners",
        "difficulty": "beginner",
        "learning_objectives": [
            "Understand Python variables and data types",
            "Master basic control flow",
            "Write simple Python programs"
        ]
    }

print("Course Metadata:")
print(f"  ID: {course_data.get('id', 'N/A')}")
print(f"  Title: {course_data.get('title', 'N/A')}")
print(f"  Difficulty: {course_data.get('difficulty', 'N/A')}")
print(f"  Learning Objectives:")
for obj in course_data.get('learning_objectives', []):
    print(f"    - {obj}")

Course Metadata:
  ID: python_basics
  Title: Python Basics for Beginners
  Difficulty: beginner
  Learning Objectives:
    - Understand Python variables and data types
    - Master basic control flow
    - Write simple Python programs


## 4. Test Cases

### Test 1: Typical Student Name

In [6]:
print("Test 1: Typical Indonesian Name")
print("="*50)

result = ai_greeting.generate_greeting("Budi", course_data)
print(f"\nResponse:\n{result}")

Test 1: Typical Indonesian Name

Response:
Halo Budi! Selamat datang di Python Basics for Beginners. Ayok kita mulai, pasti seru!


### Test 2: Different Names

In [10]:
print("Test 2: Various Names")
print("="*50)

names = ["Siti Rahayu", "Ahmad Fauzi", "Dewi"]

for name in names:
    result = ai_greeting.generate_greeting(name, course_data)
    print(f"\nName: {name}")
    print(f"Response: {result}...")

Test 2: Various Names

Name: Siti Rahayu
Response: Hi Siti Rahayu! Senang bertemu kamu di kelas Python Basics for Beginners. Mari kita mulai petualangan ini dengan semangat!...

Name: Ahmad Fauzi
Response: Halo Ahmad Fauzi! Selamat datang di Python Basics for Beginners. Mari kita mulai petualangan ini dengan semangat!...

Name: Dewi
Response: Selamat datang Dewi! Siap belajar Python Basics for Beginners? Ayok kita mulai, pasti seru!...


### Test 3: Edge Cases

In [8]:
print("Test 3: Edge Cases")
print("="*50)

# Test with empty name
print("\n3a. Empty name:")
result = ai_greeting.generate_greeting("", course_data)
print(f"Response: {result}")

# Test with minimal course metadata
print("\n3b. Minimal metadata:")
minimal_data = {"title": "Test Course", "learning_objectives": []}
result = ai_greeting.generate_greeting("Budi", minimal_data)
print(f"Response: {result}")

Test 3: Edge Cases

3a. Empty name:
Response: Hi ! Senang bertemu kamu di kelas Python Basics for Beginners. Kita belajar bersama ya, {student_name}!

3b. Minimal metadata:
Response: Selamat datang Budi! Siap belajar Test Course? Kita belajar bersama ya, {student_name}!


## 5. Analysis

In [9]:
print("Analysis Summary")
print("="*50)

# Generate multiple greetings for variety analysis
test_names = ["Budi", "Siti", "Ahmad", "Dewi", "Rudi"]
responses = []

print("\\n🔍 Testing greeting variety...")
for name in test_names:
    result = ai_greeting.generate_greeting(name, course_data)
    responses.append(result)

# Check patterns
print("\\n✅ All responses generated successfully" if all(responses) else "\\n❌ Some responses failed")

# Check for variety (not all same)
unique_responses = len(set(responses))
variety_score = f"{unique_responses}/{len(responses)}/100"
print(f"📊 Variety Score: {variety_score} (higher is better)")

# Check language quality
print("\\n🔍 Language Quality Checks:")
indo_words = ["selamat", "datang", "belajar", "kursus", "siap", "memulai", "semangat"]
for i, resp in enumerate(responses):
    # Check for Indonesian
    has_indo = any(word in resp.lower() for word in indo_words)
    
    # Check for personalization (student name appears)
    is_personalized = test_names[i].lower() in resp.lower()
    
    # Check for engaging (open-ended question, not just statement)
    is_engaging = "?" in resp or "apa" in resp.lower() or "siap" in resp.lower()
    
    # Check length efficiency
    is_efficient = 30 < len(resp) < 300
    
    status = "✅" if all([has_indo, is_personalized, is_engaging, is_efficient]) else "⚠️"
    print(f"  {status} {test_names[i]}: ID={has_indo}, Pers={is_personalized}, Eng={is_engaging}, Len={len(resp)}")

Analysis Summary
\n🔍 Testing greeting variety...
\n✅ All responses generated successfully
📊 Variety Score: 5/5/100 (higher is better)
\n🔍 Language Quality Checks:
  ⚠️ Budi: ID=True, Pers=True, Eng=False, Len=115
  ⚠️ Siti: ID=True, Pers=True, Eng=False, Len=107
  ⚠️ Ahmad: ID=True, Pers=True, Eng=False, Len=116
  ⚠️ Dewi: ID=True, Pers=True, Eng=False, Len=106
  ⚠️ Rudi: ID=True, Pers=True, Eng=False, Len=107


## 6. Notes

**Expected Output:**
- Personalized greeting using student name (minimum 1x in sentence)
- Engaging question about student's readiness
- References course content and learning objectives
- Warm, conversational Indonesian tone (bukan "Halo" saja)
- 2-4 sentences max (token efficient)

**Variety Examples:**
- "Halo Budi! Senang bertemu kamu di kelas Python Basics for Beginners. Selamat belajar!"
- "Halo Siti! Apa kamu sudah siap untuk memulai perjalanan belajar Python?"

**Fallback Behavior:**
- If API fails: Returns encouraging message with random greeting style
- Example: "Halo Andi! Selamat datang di Kursus Python. Mari kita mulai belajar!"

**Chain Behavior:**
- Post-processes LLM response to ensure quality
- Falls back to pre-built greetings if LLM response is too generic/short
- Uses randomization for variety